# Bivariate analysis: looking at two variables together

_Data Wrangling_

**Pair up columns to find relationships, group differences, and the features that actually relate to your target.**

Univariate analysis answers "what does this one column look like?" Bivariate analysis
       answers the question that actually matters for modeling: "do these two columns relate?" That is
       the whole point of this step &mdash; it is where you discover which features carry signal about the
       target, and which features are really duplicates of each other.

       The key move is to stop thinking "what chart do I want?" and start thinking "what two data types am
       I holding?" Once you classify each of the two columns as numeric or categorical, there are only three
       combinations, and each combination has a small, settled set of right pictures. The type pair chooses the
       tool.

---

This notebook is a practice scaffold for the **Bivariate analysis: looking at two variables together** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — pandas + seaborn

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

tips = sns.load_dataset("tips")   # bundled: total_bill, tip, sex, smoker, day, time, size

# === 1. NUMERIC x NUMERIC: scatter + trend line ========================
# regplot draws the scatter AND a fitted regression line in one call.
# alpha fights overplotting; lowess=True would draw a smoother instead.
sns.regplot(data=tips, x="total_bill", y="tip",
            scatter_kws={"alpha": 0.4}, line_kws={"color": "C1"})
plt.title("tip vs total_bill (trend line)")
plt.show()
# Pearson correlation: a single number for the strength/direction of the trend.
print("corr(total_bill, tip) =", round(tips["total_bill"].corr(tips["tip"]), 3))

# === 2. NUMERIC x CATEGORICAL: box plot per group + group means ========
# Split the numeric (total_bill) into one distribution per category (day).
sns.boxplot(data=tips, x="day", y="total_bill")
plt.title("total_bill by day")
plt.show()
print(tips.groupby("day", observed=True)["total_bill"].mean().round(2))

# === 3. CATEGORICAL x CATEGORICAL: normalized crosstab + grouped bars ===
# Count combinations, then normalize each ROW to a proportion so groups of
# different size are comparable (each day's smoker mix sums to 1.0).
ct = pd.crosstab(tips["day"], tips["smoker"], normalize="index")
print(ct.round(3))
#         smoker     No    Yes
# day
# Thur            0.62   0.38
# ...
ct.plot(kind="bar")        # grouped bars: smoker vs non-smoker share per day
plt.ylabel("share of tables")
plt.title("smoker share by day (row-normalized)")
plt.show()

## Visualize the data & results

_Wine chemistry: do two features (flavanoids and total phenols) relate to each other AND to the grape cultivar (the class/target)?_

In [ ]:
import numpy as np, pandas as pd
from sklearn.datasets import load_wine

df = load_wine(as_frame=True).frame
names = load_wine().target_names
x, y = "flavanoids", "total_phenols"

# Feature x feature: how tightly do the two numerics move together?
print("corr:", round(df[x].corr(df[y]), 3))          # -> 0.865

# Feature x target: group means show each cultivar's level on both features.
print(df.groupby("target")[[x, y]].mean().round(3))
#         flavanoids  total_phenols
# 0            2.982          2.840
# 1            2.081          2.259
# 2            0.781          1.679

# Subsample <=18 points per class for a readable scatter (colored by target).
rng = np.random.default_rng(7)
for cls in sorted(df["target"].unique()):
    sub = df[df["target"] == cls]
    idx = rng.choice(sub.index, size=min(18, len(sub)), replace=False)
    pts = [[round(float(df.loc[i, x]), 3), round(float(df.loc[i, y]), 3)] for i in idx]
    print(names[cls], pts)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You scatter monthly_revenue against ad_spend and see a clean upward line. A teammate writes "ad spend drives revenue." What is wrong, and what would you check?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Separate what the scatter shows (the two move together) from what it does not show (which causes which). — _A scatter measures association, not causation; the arrow of cause could run either way or come from neither._
- Name a plausible confounder &mdash; e.g. seasonality: busy months get both more ad budget and more revenue. — _A hidden third variable can drive both columns and manufacture the trend._
- Split or color the scatter by the suspected confounder (e.g. by quarter) and re-check whether the trend survives within each group. — _If the within-group trend weakens or reverses, the original line was confounded (Simpson's paradox)._

**Answer:** The scatter only shows association, not causation, so "drives" is unjustified. A confounder like seasonality could inflate both ad spend and revenue in busy months. Color the points by quarter (or another candidate confounder) and see whether the upward trend holds inside each group before claiming any causal link.

</details>

**Problem 2.** You have age (number) and plan (free/basic/pro, a category). Which bivariate views fit, and which would be wrong?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Classify the pair: one numeric (age), one categorical (plan) &rarr; this is the numeric &times; categorical case. — _The type pair, not taste, picks the chart family._
- Choose split-by-category views: a box or violin plot of age per plan, or a table of group means of age by plan. — _You are comparing one number's distribution across the categories._
- Reject a plain scatter of age vs plan and a crosstab. — _A scatter needs two numeric axes; a crosstab is for two categoricals. Neither matches this pair._

**Answer:** It is numeric &times; categorical, so split the number by the category: a box or violin plot of age per plan, or df.groupby("plan")["age"].mean(). A scatter (needs two numbers) and a crosstab (needs two categories) are the wrong tools here.

</details>

**Problem 3.** A crosstab of region &times; churned shows raw counts: region A has 900 churned, region B has 100. Your colleague concludes region A churns more. Why is the raw count misleading, and what fixes it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Ask how big each region is, not just the churn count. — _Comparing groups of very different size on raw counts is a classic trap &mdash; a big group has more of everything._
- Normalize the crosstab to proportions: pd.crosstab(region, churned, normalize="index"). — _Row-normalizing gives each region its own churn rate, which is comparable across sizes._
- Compare the rates: if A has 9,000 customers (10&percnt; churn) and B has 500 (20&percnt; churn), B actually churns more. — _Proportions, not counts, answer the real question once group sizes differ._

**Answer:** Raw counts confound churn with region size &mdash; a large region racks up more churned customers even at a lower rate. Use pd.crosstab(region, churned, normalize="index") to get each region's churn rate; on the rates, the smaller region can easily be the worse one.

</details>